# SinGAN

Train a generative model on a single image and run all SinGAN applications: random samples, harmonization, editing, paint-to-image, animation, and super resolution.

**Before running anything:** set `ENV` and `NOT_CUDA` in the **Configuration** cell below, then run all cells in order.

---
## Configuration

In [ ]:
ENV      = 'colab'  # 'colab' or 'local'
NOT_CUDA = False    # True if running locally without a GPU

In [ ]:
import os, shutil, subprocess, shlex, time
from pathlib import Path
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE_DIR = '/content/SinGAN' if ENV == 'colab' else os.path.abspath('.')
_nc      = '--not_cuda' if NOT_CUDA else ''

---
## Setup

In [ ]:
if ENV == 'colab':
    !git clone https://github.com/owaisali246/SinGAN.git $BASE_DIR

os.chdir(BASE_DIR)
!pip install -r requirements.txt -q

---
## Training

Trained models are saved to `TrainedModels/<image_name>/scale_factor=<f>,alpha=<n>/`.

> Use the **same `--max_size`** for all downstream inference commands on a given model.

### Single Image

In [ ]:
INPUT_NAME = 'starry_night.png'
MAX_SIZE   = 256
NITER      = 2000   # reduce to 1000 for ~2x speedup

cmd = f'python main_train.py --input_name {INPUT_NAME} --max_size {MAX_SIZE} --niter {NITER} {_nc}'
!{cmd}

### Parallel Training

Trains multiple images simultaneously with live side-by-side log output.
Limit to **5–6 jobs** on a 12 GB RAM machine to avoid OOM.

In [ ]:
IMAGES   = ['balloons.png', 'colusseum.png', 'mountains3.png']
MAX_SIZE = 256
NITER    = 2000

jobs = [{
    'name':     img[:-4],
    'cmd':      f'python main_train.py --input_name {img} --max_size {MAX_SIZE} --niter {NITER} {_nc}',
    'log_file': f'{img[:-4]}_train.log'
} for img in IMAGES]

for job in jobs:
    print(f"Launching {job['name']}...")
    job['log_handle'] = open(job['log_file'], 'w')
    job['proc'] = subprocess.Popen(
        shlex.split(job['cmd']),
        stdout=job['log_handle'],
        stderr=subprocess.STDOUT,
        text=True
    )
    time.sleep(10)  # stagger to avoid CUDA init race

def _tail(path, n=25):
    try:
        lines = open(path).readlines()
        return lines[-n:] if lines else ['Waiting...']
    except FileNotFoundError:
        return ['Log not found']

def _show_logs():
    html = "<div style='display:flex;gap:12px'>"
    for job in jobs:
        text = ''.join(_tail(job['log_file'])).replace('\n', '<br>').replace(' ', '&nbsp;')
        html += (
            "<div style='flex:1;border:1px solid #ddd;border-radius:4px;"
            "padding:10px;max-height:400px;overflow-y:auto;background:#f9f9f9'>"
            f"<b style='color:#1a73e8'>{job['name']}</b><br>"
            f"<pre style='font-size:11px'>{text}</pre></div>"
        )
    html += '</div>'
    clear_output(wait=True)
    display(HTML(html))

while True:
    _show_logs()
    if all(j['proc'].poll() is not None for j in jobs):
        break
    time.sleep(5)

for job in jobs:
    job['log_handle'].close()
print('All training complete.')

---
## Utilities

In [ ]:
def show_images(folder, max_images=10, cols=5):
    """Display a grid of images from a folder."""
    paths = sorted(Path(folder).glob('*.png'))[:max_images]
    if not paths:
        print(f'No images found in {folder}')
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(mpimg.imread(p))
        ax.set_title(p.name, fontsize=8)
        ax.axis('off')
    for ax in axes[len(paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def image_sizes(folder):
    """Print filename and dimensions for all PNGs in a folder."""
    from PIL import Image
    for p in sorted(Path(folder).glob('*.png')):
        img = Image.open(p)
        print(f'{p.name:<12}: {img.size[0]}x{img.size[1]}')

---
## Applications

All commands below load an already-trained model. Ensure `MAX_SIZE` matches the value used during training.

**Injection scale** controls how aggressively the model reworks a reference:
- `1` (coarsest) — heavy rework: color, texture, and structure all change
- `N−1` (finest) — subtle: only fine texture is adapted, structure is preserved

### Random Samples

Output: `Output/RandomSamples/<image>/gen_start_scale=<n>/`

In [ ]:
INPUT_NAME      = 'starry_night.png'
GEN_START_SCALE = 0    # 0 = most diverse; higher = closer to training image structure
MAX_SIZE        = 256

cmd = f'python random_samples.py --input_name {INPUT_NAME} --mode random_samples --gen_start_scale {GEN_START_SCALE} --max_size {MAX_SIZE} {_nc}'
!{cmd}

In [ ]:
show_images(f'Output/RandomSamples/{INPUT_NAME[:-4]}/gen_start_scale={GEN_START_SCALE}')

### Random Samples — Arbitrary Sizes

Output: `Output/RandomSamples_ArbitrarySizes/<image>/`

In [ ]:
INPUT_NAME = 'starry_night.png'
SCALE_H    = 1.5    # horizontal scaling factor (>1 = wider)
SCALE_V    = 1.0    # vertical scaling factor
MAX_SIZE   = 256

cmd = f'python random_samples.py --input_name {INPUT_NAME} --mode random_samples_arbitrary_sizes --scale_h {SCALE_H} --scale_v {SCALE_V} --max_size {MAX_SIZE} {_nc}'
!{cmd}

### Harmonization

Blends a pasted object into a background so it matches the background's texture and appearance.

**Setup:** place these two files under `Input/Harmonization/` before running:
- `<ref_name>.png` — naively composited image (object pasted onto background)
- `<ref_name>_mask.png` — binary mask (white = object region to harmonize)

Output: `Output/Harmonization/<input>/<ref>_out/`

In [ ]:
INPUT_NAME  = 'cows.png'   # background image — must have a trained model
REF_NAME    = 'cows.png'   # composited image in Input/Harmonization/
START_SCALE = 1
MAX_SIZE    = 256

cmd = f'python harmonization.py --input_name {INPUT_NAME} --ref_name {REF_NAME} --harmonization_start_scale {START_SCALE} --max_size {MAX_SIZE} {_nc}'
!{cmd}

### Editing

Adapts a manually edited image so that edits look consistent with the original's texture and style.

**Setup:** place these two files under `Input/Editing/` before running:
- `<ref_name>.png` — edited image
- `<ref_name>_mask.png` — binary mask marking the edited region

Output: `Output/Editing/<input>/<ref>_out/`

In [ ]:
INPUT_NAME  = 'stone.png'
REF_NAME    = 'stone.png'
START_SCALE = 1
MAX_SIZE    = 256

cmd = f'python editing.py --input_name {INPUT_NAME} --ref_name {REF_NAME} --editing_start_scale {START_SCALE} --max_size {MAX_SIZE} {_nc}'
!{cmd}

### Paint to Image

Converts a coarse paint or sketch into a photorealistic image using the trained model's texture.

**Setup:** save your paint under `Input/Paint/`.

Output: `Output/Paint2image/<input>/<ref>_out/`

In [ ]:
INPUT_NAME        = 'cows.png'
REF_NAME          = 'cows.png'   # paint image in Input/Paint/
PAINT_START_SCALE = 1
MAX_SIZE          = 256
QUANTIZE          = False        # re-trains injection scale on color-quantized output for sharper boundaries

q_flag = '--quantization_flag True' if QUANTIZE else ''
cmd = f'python paint2image.py --input_name {INPUT_NAME} --ref_name {REF_NAME} --paint_start_scale {PAINT_START_SCALE} --max_size {MAX_SIZE} {q_flag} {_nc}'
!{cmd}

### Animation

Generates a looping GIF by smoothly walking through the generators' noise space.

> **Note:** This triggers a **separate training run** (`animation_train` mode with noise padding). It cannot reuse a standard trained model — plan for the extra training time.

Output: `Output/Animation/<image>/`

In [ ]:
INPUT_NAME = 'starry_night.png'

cmd = f'python animation.py --input_name {INPUT_NAME} {_nc}'
!{cmd}

### Super Resolution

Upscales a low-resolution image using a SinGAN model trained on the LR image itself.

Output: `Output/SR/<sr_factor>/`

In [ ]:
INPUT_NAME = 'starry_night.png'
SR_FACTOR  = 4

cmd = f'python SR.py --input_name {INPUT_NAME} --sr_factor {SR_FACTOR} {_nc}'
!{cmd}

---
## SIFID Benchmark

Measures how closely generated samples match the real image's deep feature distribution (lower = better).
Run training and random sample generation before this section.

In [ ]:
IMAGE_NAME      = 'starry_night'
GEN_START_SCALE = 0
N_SAMPLES       = 50

real_dir = f'SIFID/real_images/{IMAGE_NAME}'
fake_dir = f'Output/RandomSamples/{IMAGE_NAME}/gen_start_scale={GEN_START_SCALE}'

os.makedirs(real_dir, exist_ok=True)
for i in range(N_SAMPLES):
    shutil.copy(f'Input/Images/{IMAGE_NAME}.png', f'{real_dir}/{i}.png')
print(f'Prepared {N_SAMPLES} real image copies → {real_dir}/')

In [ ]:
cmd = f'python SIFID/sifid_score.py --path2real {real_dir} --path2fake {fake_dir} --images_suffix png'
!{cmd}